# Zero-shot vs few-shot: when examples actually help

**The Agent Build Log · Build #010 · Prompt Engineering 2/8**

[Build #009](../01-anatomy-of-a-prompt/) ended with a prompt that labels support tickets correctly 75% of the time
on held-out tickets, missing only on edge cases where two rules in my rubric pointed different ways.
The usual advice is "add examples". This build tests that against the alternative: **fix the rules**.

1. [Data: examples from dev, scores on fresh tickets](#1-data-examples-from-dev-scores-on-fresh-tickets)
2. [Eight prompts](#2-eight-prompts)
3. [Run them](#3-run-them)
4. [Do examples replace a rubric?](#4-do-examples-replace-a-rubric)
5. [Skewed examples: all "high"](#5-skewed-examples-all-high)
6. [Findings](#6-findings)

**Model:** Claude Haiku 5.5 with thinking off. **Needs:** an Anthropic API key. A full run costs about **7 cents**.

In [1]:
import json, os, re, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "output": 0.50}   # $ per 1M tokens (prompts up to 100K tokens)
client = anthropic.Anthropic().with_options(max_retries=6)
DATA = json.loads(Path("data/tickets.json").read_text())
FIELDS = DATA["fields"]
DEV = {t["id"]: t for t in DATA["dev"]}
EVAL = {"fresh": DATA["fresh"], "test_009": DATA["test_009"]}
print(f"model: {MODEL} | dev {len(DEV)} | fresh {len(EVAL['fresh'])} | test_009 {len(EVAL['test_009'])}")

model: claude-haiku-5-5 | dev 20 | fresh 20 | test_009 12


## 1. Data: examples from dev, scores on fresh tickets

- **dev (20):** the tickets from build #004. Every few-shot example comes from here, so no example is ever scored.
- **fresh (20):** new tickets, labelled with build #009's rubric **before any model run**. This is the main score.
- **test_009 (12):** build #009's held-out set. I wrote rubric v2's tie-breaks after seeing its 3 misses, so for v2
  this set is **not** held out any more. It's shown for transparency; judge v2 on **fresh**.

In [2]:
pd.DataFrame([{**t["label"], "ticket": t["text"]} for t in DATA["fresh"][:6]])

,category,priority,sentiment,ticket
0,bug,urgent,negative,Checkout fails for every customer using PayPal since 9am. We're losing orders.
1,billing,low,neutral,Could you send me last month's invoice again? I can't find the email.
2,billing,medium,neutral,I was charged $19.99 instead of $14.99 for ORD-62210. Please refund the diff...
3,shipping,high,negative,My order ORD-70019 shows delivered but the box isn't here and my neighbours ...
4,account,medium,negative,The mobile app logs me out every few minutes. I can still use the website.
5,feature_request,low,positive,"Your new onboarding guide is brilliant, thank you!"


## 2. Eight prompts

All eight keep build #009's role and JSON format line. They differ in the **rubric** (none, v1 from #009, or v2 with
three tie-breaks) and the **examples** (0, 2, 4 or 8 labelled dev tickets).

In [3]:
ROLE = "You are the support triage lead at an online electronics shop. You sort incoming tickets so the right team picks them up first."
TASK = "Triage this support ticket: give its category, priority and sentiment."
FORMAT = ("Reply with only a JSON object, no other text:\n"
          '{"category": one of ' + json.dumps(FIELDS["category"]) + ', "priority": one of ' + json.dumps(FIELDS["priority"])
          + ', "sentiment": one of ' + json.dumps(FIELDS["sentiment"]) + "}")
RUBRIC_V1 = '''How our support team labels tickets:
Category
- billing: charges, refunds, plans, pricing questions, cancellations
- bug: something in the product is broken or wrong
- account: logging in, security, account settings, users
- shipping: delivery, tracking, damaged or wrong items
- feature_request: ideas for the product, and general feedback or praise
Priority
- urgent: many users blocked or money being lost right now, or a security incident (someone else in the account)
- high: one customer blocked with no workaround (locked out, charged twice for the same order, item lost or damaged,
  a crash on a main screen), or a business deadline at risk
- medium: a real problem with a workaround or a simple fix (wrong amount, coupon, cancellation, late tracking,
  exchange, a non-blocking bug), or a feature they would use right away
- low: questions, "someday" ideas, praise
Sentiment
- negative: frustration, or reports something broken, lost, damaged or blocked for them
- neutral: a calm request or question, including calm billing requests
- positive: praise or thanks is the main point'''   # identical to build #009

TIE_BREAKS = '''When two rules disagree:
- A calm request about a billing mistake is neutral, even though something went wrong.
- A bug the customer calls minor or small is still negative: it reports something broken.
- If the customer has a working alternative (another device, the website), it's medium, even if one path is blocked.'''
RUBRIC_V2 = RUBRIC_V1 + "\n" + TIE_BREAKS

def example_block(ids):
    parts = [f"<example>\n<ticket>{DEV[i]['text']}</ticket>\n<labels>{json.dumps(DEV[i]['label'])}</labels>\n</example>" for i in ids]
    return "Examples of tickets labelled by our team:\n" + "\n".join(parts)

SHOTS = {0: [], 2: [1, 7], 4: [1, 7, 12, 3], 8: [1, 7, 12, 3, 4, 9, 19, 11]}
SKEWED = [1, 4, 5, 15]   # four dev tickets that are all priority "high"

VARIANTS = {
    "0-shot, rubric v1 (#009 prompt)": (RUBRIC_V1, SHOTS[0]),
    "2-shot + rubric v1":              (RUBRIC_V1, SHOTS[2]),
    "4-shot + rubric v1":              (RUBRIC_V1, SHOTS[4]),
    "8-shot + rubric v1":              (RUBRIC_V1, SHOTS[8]),
    "0-shot, no rubric":               (None, SHOTS[0]),
    "8-shot, no rubric":               (None, SHOTS[8]),
    "4-shot all 'high' + rubric v1":   (RUBRIC_V1, SKEWED),
    "0-shot, rubric v2 (tie-breaks)":  (RUBRIC_V2, SHOTS[0]),
}

def build(variant, ticket):
    rubric, shots = VARIANTS[variant]
    parts = ([rubric] if rubric else []) + ([example_block(shots)] if shots else []) + [TASK, f"<ticket>\n{ticket}\n</ticket>", FORMAT]
    return "\n\n".join(parts)

for k, ids in {**SHOTS, "skewed": SKEWED}.items():
    print(k, Counter(DEV[i]["label"]["priority"] for i in ids) if ids else "-")
print("\n--- 2-shot prompt for one fresh ticket ---\n" + build("2-shot + rubric v1", DATA["fresh"][0]["text"])[-900:])

0 -
2 Counter({'high': 1, 'medium': 1})
4 Counter({'high': 1, 'medium': 1, 'urgent': 1, 'low': 1})
8 Counter({'medium': 3, 'high': 2, 'low': 2, 'urgent': 1})
skewed Counter({'high': 4})

--- 2-shot prompt for one fresh ticket ---

Examples of tickets labelled by our team:
<example>
<ticket>Hi, I was charged twice for order ORD-48213. Please refund the extra $49.99. Thanks, jane.doe@example.com</ticket>
<labels>{"category": "billing", "priority": "high", "sentiment": "neutral"}</labels>
</example>
<example>
<ticket>The export button gives a blank CSV when I pick more than 30 days. Not urgent, just annoying.</ticket>
<labels>{"category": "bug", "priority": "medium", "sentiment": "negative"}</labels>
</example>

Triage this support ticket: give its category, priority and sentiment.

<ticket>
Checkout fails for every customer using PayPal since 9am. We're losing orders.
</ticket>

Reply with only a JSON object, no other text:
{"category": one of ["billing", "bug", "account", "shipping", "f

## 3. Run them

8 prompts × 32 tickets × 3 runs = 768 calls, 8 at a time. Parsing is the same as build #009: a reply is **usable**
only if a program can read exactly one allowed value per field.

In [4]:
def parse(text: str):
    body = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    m = re.search(r"\{.*\}", body, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            out = {f: str(d.get(f, "")).strip().lower() for f in FIELDS}
            if all(out[f] in FIELDS[f] for f in FIELDS):
                return out
        except json.JSONDecodeError:
            pass
    return None

def run_one(variant, split, ticket, run):
    r = client.messages.create(model=MODEL, max_tokens=300, thinking={"type": "disabled"}, system=ROLE,
                               messages=[{"role": "user", "content": build(variant, ticket["text"])}])
    got = parse("".join(b.text for b in r.content if b.type == "text"))
    row = {"variant": variant, "split": split, "id": ticket["id"], "run": run, "usable": got is not None,
           "input": r.usage.input_tokens, "output": r.usage.output_tokens,
           "cost": (r.usage.input_tokens * PRICE["input"] + r.usage.output_tokens * PRICE["output"]) / 1e6}
    for f in FIELDS:
        row[f"pred_{f}"] = got[f] if got else None
        row[f] = bool(got) and got[f] == ticket["label"][f]
    return row

jobs = [(v, s, t, run) for v in VARIANTS for s, ts in EVAL.items() for t in ts for run in (1, 2, 3)]
with ThreadPoolExecutor(8) as ex:
    runs = pd.DataFrame(list(ex.map(lambda j: run_one(*j), jobs)))
runs["all3"] = runs["category"] & runs["priority"] & runs["sentiment"]
print(f"{len(runs)} calls | total cost ${runs['cost'].sum():.4f}")

def summary(df):
    return pd.Series({"all 3 right %": round(100 * df["all3"].mean()), "right / runs": f"{int(df['all3'].sum())}/{len(df)}",
                      "priority %": round(100 * df["priority"].mean()), "sentiment %": round(100 * df["sentiment"].mean()),
                      "category %": round(100 * df["category"].mean()), "usable %": round(100 * df["usable"].mean()),
                      "input tokens": round(df["input"].mean()), "$ per 1,000 tickets": round(df["cost"].mean() * 1000, 3)})

table = runs.groupby(["split", "variant"], sort=False).apply(summary).reset_index()
table[table.split == "fresh"]

768 calls | total cost $0.0705


,split,variant,all 3 right %,right / runs,priority %,sentiment %,category %,usable %,input tokens,"$ per 1,000 tickets"
0,fresh,"0-shot, rubric v1 (#009 prompt)",78,47/60,95,93,90,100,557,0.070
2,fresh,2-shot + rubric v1,82,49/60,92,100,90,100,760,0.091
4,fresh,4-shot + rubric v1,85,51/60,95,100,90,100,930,0.108
6,fresh,8-shot + rubric v1,80,48/60,90,100,90,100,1264,0.141
8,fresh,"0-shot, no rubric",42,25/60,68,77,85,100,199,0.034
10,fresh,"8-shot, no rubric",60,36/60,80,92,85,100,906,0.105
12,fresh,4-shot all 'high' + rubric v1,80,48/60,95,95,90,100,921,0.107
14,fresh,"0-shot, rubric v2 (tie-breaks)",85,51/60,100,95,90,100,651,0.080


In [5]:
table[table.split == "test_009"]

,split,variant,all 3 right %,right / runs,priority %,sentiment %,category %,usable %,input tokens,"$ per 1,000 tickets"
1,test_009,"0-shot, rubric v1 (#009 prompt)",75,27/36,92,83,100,100,554,0.070
3,test_009,2-shot + rubric v1,86,31/36,86,100,100,100,757,0.090
5,test_009,4-shot + rubric v1,86,31/36,86,100,100,100,927,0.107
7,test_009,8-shot + rubric v1,92,33/36,92,100,100,100,1261,0.141
9,test_009,"0-shot, no rubric",44,16/36,75,78,83,100,196,0.034
11,test_009,"8-shot, no rubric",58,21/36,75,92,83,100,903,0.105
13,test_009,4-shot all 'high' + rubric v1,81,29/36,92,89,100,100,918,0.106
15,test_009,"0-shot, rubric v2 (tie-breaks)",100,36/36,100,100,100,100,648,0.080


## 4. Do examples replace a rubric?

Same 8 examples, with and without the written rules, on the fresh tickets:

In [6]:
fresh = table[table.split == "fresh"].set_index("variant")
cmp = fresh.loc[["0-shot, no rubric", "8-shot, no rubric", "0-shot, rubric v1 (#009 prompt)", "8-shot + rubric v1", "0-shot, rubric v2 (tie-breaks)"],
                ["all 3 right %", "right / runs", "priority %", "sentiment %", "input tokens", "$ per 1,000 tickets"]]
cmp

,all 3 right %,right / runs,priority %,sentiment %,input tokens,"$ per 1,000 tickets"
variant,,,,,,
"0-shot, no rubric",42,25/60,68,77,199,0.034
"8-shot, no rubric",60,36/60,80,92,906,0.105
"0-shot, rubric v1 (#009 prompt)",78,47/60,95,93,557,0.070
8-shot + rubric v1,80,48/60,90,100,1264,0.141
"0-shot, rubric v2 (tie-breaks)",85,51/60,100,95,651,0.080


In [7]:
# which fresh tickets each prompt still gets wrong (out of 3 runs)
fr = runs[runs.split == "fresh"]
wrong = fr.groupby(["variant", "id"], sort=False)["all3"].sum().unstack(fill_value=0)
text_of = {t["id"]: t for t in DATA["fresh"]}
hard = [i for i in wrong.columns if (wrong[i] < 3).any()]
pd.DataFrame({i: wrong[i] for i in hard}).rename(columns=lambda i: f"{i}").T.assign(
    ticket=lambda d: [text_of[int(i)]["text"][:60] for i in d.index], label=lambda d: [text_of[int(i)]["label"] for i in d.index])

variant,"0-shot, rubric v1 (#009 prompt)",2-shot + rubric v1,4-shot + rubric v1,8-shot + rubric v1,"0-shot, no rubric","8-shot, no rubric",4-shot all 'high' + rubric v1,"0-shot, rubric v2 (tie-breaks)",ticket,label
203,3,3,3,3,0,0,3,3,I was charged $19.99 instead of $14.99 for ORD-62210. Please,"{'category': 'billing', 'priority': 'medium', 'sentiment': 'neutral'}"
205,0,0,0,0,0,0,0,0,The mobile app logs me out every few minutes. I can still us,"{'category': 'account', 'priority': 'medium', 'sentiment': 'negative'}"
206,3,3,3,3,0,0,3,3,"Your new onboarding guide is brilliant, thank you!","{'category': 'feature_request', 'priority': 'low', 'sentiment': 'positive'}"
207,3,3,3,3,0,0,3,3,Please add Apple Pay at checkout. We'd move our whole team's,"{'category': 'feature_request', 'priority': 'medium', 'sentiment': 'neutral'}"
209,0,3,3,3,0,0,0,3,"The search bar ignores accents, so 'café' finds nothing. Sma","{'category': 'bug', 'priority': 'medium', 'sentiment': 'negative'}"
210,3,3,3,3,0,3,3,3,Our invoices are due to the auditors on Thursday and the PDF,"{'category': 'bug', 'priority': 'high', 'sentiment': 'negative'}"
211,0,0,0,3,0,0,0,3,Can I change the delivery address for ORD-81120 before it sh,"{'category': 'shipping', 'priority': 'low', 'sentiment': 'neutral'}"
212,3,3,3,0,0,3,3,3,I was billed twice this month for the same subscription. Cou,"{'category': 'billing', 'priority': 'high', 'sentiment': 'neutral'}"
213,3,3,3,3,0,3,3,0,The parcel for ORD-30055 came in the wrong colour. Not a big,"{'category': 'shipping', 'priority': 'medium', 'sentiment': 'neutral'}"
214,3,1,3,0,0,0,3,3,Third time asking: where is my refund for ORD-11873?? It's b,"{'category': 'billing', 'priority': 'medium', 'sentiment': 'negative'}"


## 5. Skewed examples: all "high"

Four examples that happen to share a label is an easy mistake when you pick them by hand. How often does each prompt
answer **high** on fresh tickets whose label is **not** high?

In [8]:
not_high = fr[fr.id.map(lambda i: text_of[i]["label"]["priority"] != "high")]
skew = not_high.groupby("variant", sort=False).apply(lambda d: pd.Series({
    "said 'high' (of runs)": f"{int((d.pred_priority == 'high').sum())}/{len(d)}",
    "said 'high' %": round(100 * (d.pred_priority == "high").mean()),
})).loc[["0-shot, rubric v1 (#009 prompt)", "4-shot + rubric v1", "4-shot all 'high' + rubric v1"]]
pred_dist = fr.groupby("variant", sort=False)["pred_priority"].value_counts().unstack(fill_value=0)[FIELDS["priority"]]
print("true priority counts on fresh (x3 runs):", {p: 3 * c for p, c in Counter(t["label"]["priority"] for t in DATA["fresh"]).items()})
skew

true priority counts on fresh (x3 runs): {'urgent': 6, 'low': 18, 'medium': 21, 'high': 15}


,said 'high' (of runs),said 'high' %
variant,,
"0-shot, rubric v1 (#009 prompt)",0/45,0
4-shot + rubric v1,0/45,0
4-shot all 'high' + rubric v1,0/45,0


In [9]:
pred_dist

pred_priority,low,medium,high,urgent
variant,,,,
"0-shot, rubric v1 (#009 prompt)",15,24,15,6
2-shot + rubric v1,15,22,17,6
4-shot + rubric v1,15,24,15,6
8-shot + rubric v1,18,21,15,6
"0-shot, no rubric",24,13,14,9
"8-shot, no rubric",18,15,21,6
4-shot all 'high' + rubric v1,15,24,15,6
"0-shot, rubric v2 (tie-breaks)",18,21,15,6


## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [10]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "thinking": "disabled", "calls": len(runs), "cost": round(runs["cost"].sum(), 4),
    "table": table.to_dict("records"), "skew": skew.reset_index().to_dict("records"),
    "pred_priority_fresh": pred_dist.reset_index().to_dict("records"),
    "wrong_fresh": {str(k): v for k, v in wrong.T.to_dict().items()},
    "runs": runs.to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("calls", "cost", "skew")}, indent=1, default=str))

{
 "calls": 768,
 "cost": 0.0705,
 "skew": [
  {
   "variant": "0-shot, rubric v1 (#009 prompt)",
   "said 'high' (of runs)": "0/45",
   "said 'high' %": 0
  },
  {
   "variant": "4-shot + rubric v1",
   "said 'high' (of runs)": "0/45",
   "said 'high' %": 0
  },
  {
   "variant": "4-shot all 'high' + rubric v1",
   "said 'high' (of runs)": "0/45",
   "said 'high' %": 0
  }
 ]
}
